# <font color='blue'> Chapter 22: Building Convolutional Neural Networks with Keras </font>

In the previous chapters, we developed the mathematical foundations of Convolutional Neural Networks (CNNs). We learned how convolution extracts local features, how pooling reduces spatial dimensions, and why CNNs outperform fully connected neural networks for image analysis.

In this chapter, we implement these concepts using TensorFlow and Keras.

Rather than simply presenting code, we shall explain how each layer corresponds to the mathematical operations introduced previously.

By the end of this chapter, you will understand how to construct, train, evaluate, and interpret a Convolutional Neural Network for image classification.

---

# <font color='orange'> 1. The CNN Workflow </font>

A typical CNN follows the pipeline

```
Input Image

↓

Convolution

↓

Activation

↓

Pooling

↓

Convolution

↓

Activation

↓

Pooling

↓

Flatten

↓

Dense

↓

Output
```

Each stage transforms the image into increasingly meaningful feature representations.

---

# <font color='orange'> 2. Importing the Required Libraries </font>

We begin by importing TensorFlow and Keras.

```python
import tensorflow as tf

from tensorflow import keras

from tensorflow.keras import layers

import matplotlib.pyplot as plt

import numpy as np
```

---

# <font color='orange'> 3. Loading the Dataset </font>

We shall use the Fashion-MNIST dataset.

```python
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
```

The dataset contains

- 60,000 training images,
- 10,000 testing images,
- ten clothing categories.

Each image has dimensions

$$
28\times28.
$$

---

# <font color='orange'> 4. Data Preprocessing </font>

CNNs expect images with an explicit channel dimension.

Since Fashion-MNIST consists of grayscale images,

we reshape the data.

```python
X_train = X_train.reshape(-1,28,28,1)

X_test = X_test.reshape(-1,28,28,1)
```

The final dimension

```
1
```

represents one colour channel.

Next,

normalize the images.

```python
X_train = X_train.astype("float32")/255

X_test = X_test.astype("float32")/255
```

Pixel values now lie between

$$
0
\quad\text{and}\quad
1.
$$

---

# <font color='orange'> 5. Building the CNN </font>

```python
model = keras.Sequential([

    layers.Conv2D(
        filters=32,
        kernel_size=(3,3),
        activation="relu",
        padding="same",
        input_shape=(28,28,1)
    ),

    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(
        filters=64,
        kernel_size=(3,3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dense(
        10,
        activation="softmax"
    )

])
```

---

# <font color='orange'> 6. Understanding Conv2D </font>

Consider

```python
layers.Conv2D(
    filters=32,
    kernel_size=(3,3),
    activation="relu",
    padding="same"
)
```

Each argument has a specific meaning.

---

## filters

```python
filters=32
```

The layer learns

32 different filters.

Therefore,

the output consists of

32 feature maps.

---

## kernel_size

```python
kernel_size=(3,3)
```

Each filter has dimensions

$$
3\times3.
$$

These filters slide across the image,

computing local weighted sums.

---

## activation

```python
activation="relu"
```

After convolution,

ReLU is applied

$$
a=\max(0,z).
$$

---

## padding

```python
padding="same"
```

Zero-padding is added,

ensuring that the output has approximately the same height and width as the input.

Without padding,

the feature maps would shrink after every convolution.

---

# <font color='orange'> 7. MaxPooling2D </font>

```python
layers.MaxPooling2D(
    pool_size=(2,2)
)
```

Each

$$
2\times2
$$

region becomes

one value.

Example

```
6 4

2 9

↓

9
```

Pooling reduces computation while preserving strong activations.

---

# <font color='orange'> 8. Flatten Layer </font>

After several convolutional layers,

suppose the output shape becomes

```
7 × 7 × 64
```

Flatten converts this tensor into

```
3136-dimensional vector
```

No learning occurs.

Only the tensor shape changes.

---

# <font color='orange'> 9. Dense Layers </font>

The Dense layer performs classification.

Each neuron computes

$$
\boxed{
a=f(Wx+b).
}
$$

The final Dense layer

```python
Dense(10,
      activation="softmax")
```

produces

ten probabilities,

one for each clothing category.

---

# <font color='orange'> 10. Compiling the Model </font>

```python
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)
```

This specifies

- Adam optimizer,
- Cross-Entropy loss,
- Classification accuracy.

---

# <font color='orange'> 11. Training the CNN </font>

```python
history = model.fit(

    X_train,

    y_train,

    epochs=10,

    validation_split=0.2

)
```

During every epoch,

Keras performs

```
Forward Propagation

↓

Loss

↓

Backpropagation

↓

Parameter Update
```

exactly as discussed previously.

---

# <font color='orange'> 12. Evaluating the Model </font>

```python
test_loss,

test_accuracy = model.evaluate(

    X_test,

    y_test

)
```

This evaluates the trained network on unseen images.

---

# <font color='orange'> 13. Making Predictions </font>

```python
predictions = model.predict(X_test)
```

The prediction for one image might be

```
[0.001,

0.002,

0.981,

...

]
```

The predicted class is

```python
np.argmax(predictions[0])
```

---

# <font color='orange'> 14. Model Summary </font>

```python
model.summary()
```

Typical output

```
Layer

↓

Output Shape

↓

Parameters
```

Example

```
Conv2D

↓

(28,28,32)

↓

320
```

The summary helps verify the architecture.

---

# <font color='orange'> 15. How Many Parameters? </font>

Suppose

```
Input

28 × 28 × 1
```

First convolution

```
32 filters

3 × 3
```

Each filter contains

$$
3\times3\times1=9
$$

weights,

plus

one bias.

Parameters per filter

$$
10.
$$

Total

$$
32\times10=320.
$$

Notice

only

320 parameters

are learned,

far fewer than a fully connected layer.

---

# <font color='orange'> 16. Visualising Feature Maps </font>

After training,

we can visualize the outputs of convolutional layers.

```
Input Image

↓

Feature Map 1

↓

Feature Map 2

↓

Feature Map 3

↓

...
```

Different filters respond to

- edges,
- textures,
- corners,
- curves.

These feature maps provide insight into what the network has learned.

---

# <font color='orange'> 17. Typical CNN Architecture </font>

```
Input

↓

Conv

↓

ReLU

↓

Pooling

↓

Conv

↓

ReLU

↓

Pooling

↓

Flatten

↓

Dense

↓

Softmax
```

This basic architecture forms the foundation of many modern CNNs.

---

# <font color='red'> 18. Mathematical Foundations </font>

Each convolutional layer computes

$$
\boxed{
F
=
ReLU
(
I*K+b
),
}
$$

where

- $I$ is the input tensor,
- $K$ is the learnable filter,
- $b$ is the bias.

Pooling then reduces the spatial dimensions

$$
F
\rightarrow
P.
$$

Finally,

the Dense layer computes

$$
\boxed{
a
=
f(Wx+b),
}
$$

performing the final classification.

Thus,

the complete CNN combines

- convolution,
- activation,
- pooling,
- dense classification.

---

# <font color='orange'> 19. Common Misconceptions </font>

### Misconception 1

> More filters always improve performance.

**False.**

Increasing the number of filters increases the model's capacity but also raises computational cost and the risk of overfitting.

---

### Misconception 2

> Flatten learns features.

**False.**

Flatten simply reshapes the tensor into a vector. Feature learning occurs in the convolutional layers.

---

### Misconception 3

> Conv2D replaces Dense layers.

**False.**

Convolutional layers extract features, while Dense layers typically perform the final classification based on those features.

---

# <font color='purple'> 20. Conceptual Summary </font>

| Layer | Purpose |
|:---|:---|
| Conv2D | Learns spatial features using convolution |
| ReLU | Introduces nonlinearity |
| MaxPooling2D | Reduces spatial dimensions |
| Flatten | Converts feature maps into a vector |
| Dense | Performs classification |
| Softmax | Produces class probabilities |

> **Key Insight:** A Convolutional Neural Network combines convolutional layers, activation functions, pooling layers, and fully connected layers into a unified architecture for image classification. Convolution extracts local features, pooling reduces computational complexity, and Dense layers integrate the extracted information to produce final predictions. TensorFlow and Keras implement these mathematical operations efficiently, enabling the training of powerful computer vision models with only a few lines of code.